In [1]:
import numpy as np
import torch
import gymnasium as gym
import torch.nn.functional as F
from q_network import QNetwork

In [2]:
def obs_to_tensor(state, env, device):
    if isinstance(env.observation_space, gym.spaces.Discrete):
        return F.one_hot(
            torch.tensor(state, dtype=torch.long),
            num_classes=env.observation_space.n
        ).float().unsqueeze(0).to(device)
    else:
        return torch.as_tensor(
            state, dtype=torch.float32
        ).reshape(1, -1).to(device)

def evaluate_dqn(path, env, episodes=100, seed=42):  
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
    if isinstance(env.observation_space, gym.spaces.Discrete):
        input_dim = env.observation_space.n
    else:
        input_dim = int(np.prod(env.observation_space.shape))

    q_net = QNetwork(input_dim, env.action_space.n).to(device)
    q_net.load(path)

    episode_rewards = []
    for episode in range(episodes):
        state, info = env.reset(seed=seed + episode)

        terminated = False
        truncated = False
        reward_ep = 0.0

        while not terminated and not truncated:
            state_tensor = obs_to_tensor([state], env, device)
            action = q_net.get_action(state_tensor)
            state, reward, terminated, truncated, info = env.step(action)
            reward_ep += reward

        episode_rewards.append(reward_ep)

    mean_reward = float(np.mean(episode_rewards))
    std = float(np.std(episode_rewards))

    return mean_reward, std

In [3]:
env = gym.make("MountainCar-v0", render_mode = None)
path = "LucioLuqueMaterazzi_Agent.pth"
seed = 42
episodes= 100
rewards, std = evaluate_dqn(path, env, episodes=episodes, seed = seed)
print(f"Average reward over {episodes} episodes: {rewards} ± {round(std, 2)} std")
episodes= 1000
rewards, std = evaluate_dqn(path, env, episodes=episodes, seed = seed)
print(f"Average reward over {episodes} episodes: {rewards} ± {round(std, 2)} std")
episodes= 10000
rewards, std = evaluate_dqn(path, env, episodes=episodes, seed = seed)
print(f"Average reward over {episodes} episodes: {rewards} ± {round(std, 2)} std")

C:\Users\Lucio\AppData\Local\Temp\ipykernel_11512\21569098.py:8: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  return torch.as_tensor(


Average reward over 100 episodes: -97.25 ± 8.47 std
Average reward over 1000 episodes: -99.004 ± 8.11 std
Average reward over 10000 episodes: -99.167 ± 8.01 std
